<a href="https://colab.research.google.com/github/FishyFoshy/COMP3608-Project/blob/main/Dataset_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Cleaning of Dataset 1

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

# Load datasets
df1 = pd.read_csv('Dataset 1.csv')

# Removes rows that don't have HIGH confidence in their sales estimates
df1 = df1[(df1['saleEstimate_confidenceLevel'] == 'HIGH')]

# Drops all unnecessary columns
df1 = df1.drop(columns=['fullAddress', 'postcode', 'rentEstimate_lowerPrice', 'rentEstimate_currentPrice', 'rentEstimate_upperPrice', 
                        'saleEstimate_lowerPrice', 'saleEstimate_upperPrice', 'saleEstimate_ingestedAt', 'saleEstimate_valueChange.numericChange', 
                        'saleEstimate_valueChange.percentageChange', 'saleEstimate_valueChange.saleDate', 'history_date', 'history_price', 'history_percentageChange', 
                        'history_numericChange', 'saleEstimate_confidenceLevel'])

# Drops all with null values
df1.dropna(inplace=True)

# Separates non numerical columns from numerical ones
non_numerical_columns = ['saleEstimate_currentPrice', 'tenure', 'propertyType', 'currentEnergyRating']
numerical_features = [col for col in df1.columns if col not in non_numerical_columns]

# Scale numerical features
scaler = StandardScaler()
df1[numerical_features] = scaler.fit_transform(df1[numerical_features])

# One-Hot Encodes all categorical data
categorical_cols_to_encode = df1.select_dtypes(include='object').columns
df1 = pd.get_dummies(df1, columns=categorical_cols_to_encode, drop_first=True)

# Identifies target column from the features
target_column = 'saleEstimate_currentPrice'
features = [col for col in df1.columns if col != target_column]

x = df1[features].copy()
y = df1[target_column].copy()

# Splits the data 80/20 for training and testing the model respectfully
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print("=" * 60)
print("DATASET 1: Dataset 1.csv")
print("=" * 60)
print("\nFirst 10 rows:")
display(df1.head(10))
print("\nData types:")
print(df1.dtypes)
print("\nMissing values:")
print(df1.isnull().sum())
print("\nTarget variable (saleEstimate_currentPrice) statistics:")
print(df1['saleEstimate_currentPrice'].describe())

### Linear Regression for Dataset 1

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import matplotlib.pyplot as plt

# Train baseline Linear Regression
lr = LinearRegression()
lr.fit(x_train, y_train)
y_pred_lr = lr.predict(x_test)

# Evaluate
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print("=" * 50)
print("BASELINE: Linear Regression on Dataset 1.csv")
print("=" * 50)
print(f"RMSE:\t${rmse_lr:,.2f}")
print(f"MAE:\t${mae_lr:,.2f}")
print(f"R²:\t{r2_lr:.4f}")
print(f"\nCross-validation RMSE:\t${-cross_val_score(lr, x, y, cv=5, scoring='neg_root_mean_squared_error').mean():,.2f}")
print(f"Cross-validation MAE:\t${-cross_val_score(lr, x, y, cv=5, scoring='neg_mean_absolute_error').mean():,.2f}")
print(f"Cross-validation R²:\t{cross_val_score(lr, x, y, cv=5, scoring='r2').mean():.4f}")

# Predicted vs Actual scatter
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_lr, alpha=0.4, s=15)
plt.plot([0, y_test.max()], [0, y_test.max()], 'r--', label='Perfect prediction')
plt.xlabel('Actual Price ($)')
plt.ylabel('Predicted Price ($)')
plt.title('Baseline Linear Regression: Predicted vs Actual House Price')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns

# Fetching Feature Names and Coefficients
feature_names = x_train.columns
coefficients = lr.coef_

# Wrapping them together in a Pandas DataFrame for Easy Comparison and Viewing
feature_importance_lr = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
})

# Using the Absolute Function before Sorting the Coefficients in Descending Order
feature_importance_lr['Abs_Coefficient'] = np.abs(feature_importance_lr['Coefficient'])
top_10_features_lr = feature_importance_lr.sort_values(by='Abs_Coefficient', ascending=False).head(10)

# Plotting Graph
plt.figure(figsize=(10, 6))
sns.barplot(
    data=top_10_features_lr,
    x='Coefficient',
    y='Feature',
)
plt.title('Top 10 Features Driving Property Prices (Linear Regression)', fontsize=15)
plt.xlabel('Coefficient Value (Impact on Price)', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.grid(True, axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()
display(top_10_features_lr[['Feature', 'Coefficient']])